#### NLP Embeddings Clustering

##### Context

You are tasked with analyzing two datasets:

1. BBC News dataset: Contains text data grouped into 5 categories: Business, Entertainment, Politics, Sport, and Tech.
2. 20NewsGroups dataset: Contains text data grouped into 20 topics.

The goal is to transform the data into word embeddings, apply clustering techniques, and evaluate the results. Improve the results by using PCA for dimensionality reduction.

##### Problem Statement

Cluster the articles in both datasets using the following methods:

1. Feature extraction: Use TF-IDF, FastText, and Word2Vec to create embeddings.
2. Clustering algorithms: Implement K-Means, HDBSCAN, and Agglomerative Clustering.
3. Evaluation metrics: Evaluate the clustering results using NMI (Normalized Mutual Information), ARI (Adjusted Rand Index), AMI (Adjusted Mutual Information), and the Silhouette Score or Elbow Method.

##### Loading Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk
import spacy
import string

import hdbscan
from hdbscan import HDBSCAN
from sklearn.metrics.pairwise import cosine_distances
from sklearn.cluster import KMeans 
from sklearn.cluster import AgglomerativeClustering #Agglomerative
from scipy.cluster.hierarchy import linkage, dendrogram #Agglomerative
from scipy.cluster.hierarchy import fcluster #Agglomerative
from collections import Counter #Agglomerative_points_to_clusters
from sklearn.metrics import silhouette_score

from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from gensim.models import Word2Vec 
from sklearn.feature_extraction.text import TfidfVectorizer 
from gensim.models import FastText 

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split 
from sklearn.datasets import make_blobs 
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, normalized_mutual_info_score

###### Expanding column width to see the full sentences in dataframes

In [ ]:
pd.set_option('display.max_colwidth', None)

#### Loading Datasets

###### Loading 20newsgroups dataset

In [ ]:
#Loading good 10% of 20newsgroups
data_20newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
documents_20newsgroups = data_20newsgroups.data
labels_20newsgroups = data_20newsgroups.target
labels_names_20newsgroups = data_20newsgroups.target_names
news_dict = {
    'label': labels_20newsgroups,
    'comment': documents_20newsgroups,
}
news_df = pd.DataFrame(news_dict)
news_df['label_name'] = news_df['label'].apply(lambda x: labels_names_20newsgroups[x])

_, x_test_news, _, y_test_news = train_test_split(
    news_df,
    news_df['label'],
    test_size=0.1,
    random_state=42,
    stratify=news_df['label']
)

#Using the x_test_news
df_20newsgroups = x_test_news.copy()
#----------------------------------------------------------------------------

#Renaming columns for clarity of Column names
columns_mapping = {'label':'ground_truth_label','comment': 'sentence','label_name':'category'}
#Applying the renaming of Columns
df_20newsgroups = df_20newsgroups.rename(columns=columns_mapping)

#Cleaning Columns names (strip whitespace) & creating a relevant Columns list
df_20newsgroups.columns = df_20newsgroups.columns.str.strip()
df_20newsgroups_columns = list(df_20newsgroups.columns)
print("Cleaning Columns of 20newsgroups DataFrame & putting them in df_20newsgroups_columns list")
print(df_20newsgroups_columns)
print('\n')

#display(df_20newsgroups.head())

###### Loading bbc_news dataset

In [ ]:
#Loading all the bbc news dataset from relevant csv file
df_bbcnews_org = pd.read_csv("../data/bbc_news_test.csv")
#display(df_bbcnews.head())

df_bbcnews = df_bbcnews_org.copy()

#----------------------------------------------------------------------------

#Renaming columns for clarity of Column names
columns_mapping = {'ArticleId':'id', 'Text':'sentence','Category':'category'}

#Applying the renaming of Columns
df_bbcnews = df_bbcnews.rename(columns=columns_mapping)

#Cleaning Columns names (strip whitespace) & creating a relevant Columns list
df_bbcnews.columns = df_bbcnews.columns.str.strip()
df_bbcnews_columns = list(df_bbcnews.columns)
print("Cleaning Columns of bbc_news DataFrame & putting them in df_bbcnews_columns list")
print(df_bbcnews_columns)

#display(df_bbcnews.head())

#### EDA for Datasets

##### EDA for 20newsgroups dataframe